## A Resume Agentic AI Chatbot

An intelligent resume chatbot that answers question about career, and experience using the resume knowledge

In [ ]:
from dotenv import load_dotenv
from pypdf import PdfReader
from agents import Agent, Runner, function_tool
import gradio as gr

In [ ]:
load_dotenv(override=True)
MODEL = "gpt-4o-mini"

In [ ]:

@function_tool
def record_user_details(email: str, name: str = "Name not provided", notes: str = "not provided") -> dict:
    """Use this tool to record that a user is interested in being in touch and provided an email address.

    Args:
        email: The email address of this user
        name: The user's name, if they provided it
        notes: Any additional information about the conversation that's worth recording to give context
    """
    return {"recorded": "ok", "email": email, "name": name}

@function_tool
def record_unknown_question(question: str) -> dict:
    """Always use this tool to record any question that couldn't be answered.

    Args:
        question: The question that couldn't be answered
    """
    return {"recorded": "ok", "question": question}

In [ ]:
reader = PdfReader("me/Kayode-Ezekiel-Adeyemi-DevOps-CV.pdf")
linkedin = ""
for page in reader.pages:
    text = page.extract_text()
    if text:
        linkedin += text

In [ ]:
print(linkedin)

In [ ]:
with open("me/summary.txt", "r", encoding="utf-8") as f:
    summary = f.read()
name = "Kayode Ezekiel Adeyemi"
career_instructions = f"""You are acting as {name}. You are answering questions on {name}'s website,
particularly questions related to {name}'s career, background, skills and experience.
Your responsibility is to represent {name} for interactions on the website as faithfully as possible.
You are given a summary of {name}'s background and LinkedIn profile which you can use to answer questions.
Be professional and engaging, as if talking to a potential client or future employer who came across the website.

If you don't know the answer to any question, use your record_unknown_question tool to record the question that you couldn't answer, even if it's about something trivial or unrelated to career.

If the user is engaging in discussion, try to steer them towards getting in touch via email; ask for their email and record it using your record_user_details tool.

## Summary:
{summary}

## LinkedIn Profile:
{linkedin}

With this context, please chat with the user, always staying in character as {name}."""

career_agent = Agent(
    name="Career Assistant",
    instructions=career_instructions,
    model=MODEL,
    tools=[record_user_details, record_unknown_question]
)   

In [ ]:
async def chat(message, history):
    messages = []
    for msg in history:
        messages.append({"role": msg["role"], "content": msg["content"]})

    messages.append({"role": "user", "content": message})
    result = await Runner.run(career_agent, messages)
    return result.final_output

gr.ChatInterface(chat, type="messages").launch()
    